In [ ]:
import twstock
import yfinance as yf
import pandas as pd

def format_ticker(code: str) -> str:
    """
    判斷市場類別，回傳 xxxx.TW 或 xxxx.TWO。
    根據 twstock 的邏輯，需要檢查該代碼是上市還是上櫃。
    """
    try:
        info = twstock.codes.get(code)
        if info and info.type == '股票':
            return f"{code}.TW" if info.market == '上市' else f"{code}.TWO"
        return f"{code}.TW"
    except Exception:
        return f"{code}.TW"

def search_stock_info(keyword: str):
    """
    搜尋個股並顯示詳細資訊。支援代號或名稱關鍵字。
    """
    stocks = [v for v in twstock.codes.values() if v.type == '股票']
    results = []

    for s in stocks:
        if keyword == s.code or keyword in s.name:
            results.append(s)

    if not results:
        print(f"❌ 查無與 '{keyword}' 相符的股票資訊。")
        return

    print(f"\n🔍 找到 {len(results)} 筆相關結果：")
    print("-" * 50)
    for res in results:
        market_name = res.market
        print(f"代號: {res.code} | 名稱: {res.name} | 市場: {market_name} | 產業: {res.group}")
    print("-" * 50)

def filter_stocks_by_group(group_name: str):
    """
    依產業別列出所有相關個股。
    """
    stocks = [v for v in twstock.codes.values() if v.type == '股票']
    results = [s for s in stocks if group_name in s.group]

    if not results:
        print(f"❌ 在 '{group_name}' 產業中找不到任何股票。")
        return

    print(f"\n✅ '{group_name}' 產業下的股票清單 ({len(results)} 筆)：")
    print("-" * 50)
    for res in results:
        market_name = res.market
        print(f"代號: {res.code} | 名稱: {res.name} | 市場: {market_name}")
    print("-" * 50)

def get_stock_history(symbol: str, period: str = "1mo", start_date: str = None, end_date: str = None):
    """
    下載並整理行情 DataFrame。支援相對時間與自訂日期。
    """
    try:
        ticker = yf.Ticker(symbol)
        if start_date and end_date:
            df = ticker.history(start=start_date, end=end_date)
        else:
            df = ticker.history(period=period)

        if df.empty:
            print(f"⚠️ 無法取得 '{symbol}' 的歷史數據。")
            return None

        return df
    except Exception as e:
        print(f"❌ 下載數據時發生錯誤: {e}")
        return None

def display_stock_summary(symbol: str, df: pd.DataFrame):
    """
    計算並印出行情統計數據。
    """
    if df is None or df.empty:
        return

    print(f"\n📊 '{symbol}' 歷史行情統計摘要 ({df.index[0].date()} ~ {df.index[-1].date()})")
    print("-" * 50)
    print(f"資料筆數       : {len(df)} 筆")
    print(f"最高價 (High)  : {df['High'].max():.2f}")
    print(f"最低價 (Low)   : {df['Low'].min():.2f}")
    print(f"平均收盤價     : {df['Close'].mean():.2f}")
    print(f"最新收盤價     : {df['Close'].iloc[-1]:.2f}")
    print(f"平均成交量     : {df['Volume'].mean():.0f}")
    print("-" * 50)
    print("📈 資料預覽 (前 5 筆):")
    print(df.head())
    print("\n📉 資料預覽 (後 5 筆):")
    print(df.tail())
    print("-" * 50)

def main_cli():
    """
    CLI 互動選單主迴圈。
    """
    while True:
        print("\n==================================================")
        print("  📈 台股股票歷史數據查詢系統 (CLI 介面)")
        print("==================================================")
        print("  [1] 搜尋股票 (依代號/名稱)")
        print("  [2] 依產業別篩選股票")
        print("  [3] 抓取個股歷史行情 (yfinance)")
        print("  [4] 快速查詢並下載數據 (一鍵流程)")
        print("  [0] 離開系統")
        print("==================================================")

        choice = input("請選擇功能 [0-4]: ").strip()

        if choice == '1':
            keyword = input("请输入股票代號或名稱關鍵字: ").strip()
            search_stock_info(keyword)

        elif choice == '2':
            group = input("请输入產業別 (例如: 半導體, 金融): ").strip()
            filter_stocks_by_group(group)

        elif choice == '3':
            code = input("请输入股票代號 (例: 2330): ").strip()
            try:
                info = twstock.codes.get(code)
                if not info or info.type != '股票':
                    print("❌ 無效的股票代號，請輸入正確的數字代碼。")
                    continue
                market_suffix = ".TW" if info.market == '上市' else ".TWO"
                full_ticker = f"{code}{market_suffix}"
            except Exception:
                print("❌ 無效的股票代號，請輸入正確的數字代碼。")
                continue

            print("\n選擇時間區間:")
            print("  [1] 過去 1 個月 (1mo)")
            print("  [2] 過去 3 個月 (3mo)")
            print("  [3] 過去 6 個月 (6mo)")
            print("  [4] 過去 1 年 (1y)")
            print("  [5] 自訂日期區間")
            time_choice = input("請選擇 [1-5]: ").strip()

            df = None
            if time_choice == '1':
                df = get_stock_history(full_ticker, period="1mo")
            elif time_choice == '2':
                df = get_stock_history(full_ticker, period="3mo")
            elif time_choice == '3':
                df = get_stock_history(full_ticker, period="6mo")
            elif time_choice == '4':
                df = get_stock_history(full_ticker, period="1y")
            elif time_choice == '5':
                start_d = input("請輸入開始日期 (YYYY-MM-DD): ").strip()
                end_d = input("請輸入結束日期 (YYYY-MM-DD): ").strip()
                df = get_stock_history(full_ticker, start_date=start_d, end_date=end_d)
            else:
                print("❌ 無效的選擇。")

            if df is not None:
                display_stock_summary(full_ticker, df)

        elif choice == '4':
            code = input("请输入股票代號 (例如: 2330): ").strip()
            try:
                info = twstock.codes.get(code)
                if not info or info.type != '股票':
                    print("❌ 無效的股票代號，請輸入正確的數字代碼。")
                    continue
                market_suffix = ".TW" if info.market == '上市' else ".TWO"
                full_ticker = f"{code}{market_suffix}"

                print(f"🔍 正在處理 {info.name} ({full_ticker})...")
                df = get_stock_history(full_ticker, period="1y")
                if df is not None:
                    display_stock_summary(full_ticker, df)
            except Exception as e:
                print(f"❌ 處理過程中發生錯誤: {e}")

        elif choice == '0':
            print("👋 再見！感謝使用台股查詢系統。")
            break
        else:
            print("⚠️ 無效的選項，請重新選擇。")

if __name__ == "__main__":
    main_cli()
